In [14]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.ica_pipeline import *
from utils.epoch_and_average import *

# from scipy.stats import linregress
from statsmodels.regression.linear_model import OLS
import re
from IPython.display import clear_output
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
DATA_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/crystal"
STIMULI_EXCEL = "/Users/jowanglin/regression-based_ERP/data/stimuli/CRYSTAL_master-sheet.xlsx"

NOISE_COV = None                      # Noise covariance used for pre-whitening. If None (default), channels are scaled to unit variance (“z-standardized”) as a group by channel type prior to the whitening by PCA.
RANDOM_STATE = 42                     # Fix this throughout the script -- always!!

METHOD = "infomax"                    # MNE accepts 'fastica' | 'infomax' | 'picard'; using "picard" to match EEGLAB default -> need to pip install python-picard
FIT_PARAMS={"extended": True,         # EEGLAB default is infomax extended
            "weights": None,          # The initialized unmixing matrix. Defaults to None, which means the identity matrix is used.
            "l_rate": None,           # This quantity indicates the relative size of the change in weights. Defaults to 0.01 / log(n_features ** 2).
            "block": None,            # The block size of randomly chosen data segments. Defaults to floor(sqrt(n_times / 3.)).           
            "w_change": 1e-12,        # The change at which to stop iteration. Defaults to 1e-12.
            "anneal_deg": 60.0,       # The angle (in degrees) at which the learning rate will be reduced. Defaults to 60.0.
            "anneal_step": 0.9,       # The factor by which the learning rate will be reduced. Defaults to 0.9.
            "n_subgauss": 1,          # The number of subgaussian components. Only considered for extended Infomax. Defaults to 1.
            "kurt_size": 6000,        # The window size for kurtosis estimation. Only considered for extended Infomax. Defaults to 6000.
            "blowup": 10000}          # The maximum difference allowed between two successive estimations of the unmixing matrix. Defaults to 10000.

# for picard
#FIT_PARAMS={"tol": 1e-7,
            #"ortho": False, # If True, uses Picard-O. Otherwise, uses the standard Picard.
            #"fastica_it": None} # If an int, perform fastica_it iterations of FastICA before running Picard. It might help starting from a better point.


MAX_ITER=1000 
#allow_ref_meg -> irrelevant

CORR_THRESHOLD = 0.85
VERBOSE=True 

In [16]:
write_to_path = f"{DATA_DIR}/after_ica/subj_eog_indices.txt"
f = open(write_to_path, "w")
f.write("")
f.close()

raw_clean_list, eog_indices_list = [], []
f = open(write_to_path, "a")
for num in range(1, 28):
    raw_file_name = f"subj{str(num).zfill(3)}_reref_filt.set"
    try:
        raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{raw_file_name}", preload=True, verbose=False)
    except FileNotFoundError:
        continue
    
    print(f"subj{str(num).zfill(3)}")
    raw_reref, raw_for_ica, rank = preprocess_for_ica(raw,
                                                  standard_montage="standard_1020",
                                                  eog_channels=["HEO", "VEO"],
                                                  ref_channels=["M1", "M2"],
                                                  time_threshold=3.0,
                                                  after_event_code_buffer=1.5,
                                                  before_event_code_buffer=0.5,
                                                  rank_tol=1e-4,
                                                  verbose=False)
    raw_clean, eog_indices, _ = perform_ica(raw_to_clean=raw_reref,
                                            raw_for_ica=raw_for_ica,
                                            n_components=rank,
                                            noise_cov=NOISE_COV,
                                            method=METHOD,
                                            fit_params=FIT_PARAMS,
                                            max_iter=MAX_ITER,
                                            manual_inspection=False,
                                            corr_threshold=CORR_THRESHOLD,
                                            eog_like_channels=["VEO"], 
                                            verbose=False)
    raw_clean.save(f"{DATA_DIR}/after_ica/subj{str(num).zfill(3)}_after_ica_raw.fif", overwrite=True)
    raw_clean_list.append(raw_clean)
    
    f.write(f"subj{str(num).zfill(3)}: " + str(eog_indices) + "\n")
    eog_indices_list.append(eog_indices)
f.close()

clear_output()


In [17]:
# sanity checking channel positions are set, channel names are upper case, and that annotations are not lost
raw_check = raw_clean_list[0]

m = raw_check.get_montage()
pos = m.get_positions()["ch_pos"]
print(pos["FP1"])
display(pd.DataFrame(raw_check.annotations).head(10))

[-0.0309026   0.11458518  0.02786657]


,onset,duration,description,orig_time,extras
0,12.276,0.0,1,None,{}
1,25.898,0.0,210,None,{}
2,26.964,0.0,79,None,{}
3,27.324,0.0,220,None,{}
4,27.684,0.0,220,None,{}
5,28.044,0.0,220,None,{}
6,28.404,0.0,220,None,{}
7,28.764,0.0,220,None,{}
8,29.124,0.0,220,None,{}
9,29.484,0.0,79,None,{}


In [12]:
print("IC indices dropped for each subject:")
write_to_path = f"{DATA_DIR}/after_ica/subj_eog_indices.txt"
f = open(write_to_path, "r")
print(f.read())
f.close()

IC indices dropped for each subject:
subj001: []
subj003: []
subj004: []
subj005: []
subj006: []
subj007: []
subj008: []
subj009: []
subj010: []
subj011: []
subj012: []
subj014: []
subj017: []
subj018: []
subj020: []
subj021: []
subj022: []
subj024: []
subj025: []
subj027: []



### EEGLAB-style bin-based epoching, words 1-7
- (-100 ms, 700 ms)
- (-100 ms, 0) baseline correction (mode = mean)
- maximum peak-to-peak rejection threshold (all channels): 200 uV

In [3]:
df_stimuli = pd.read_excel(STIMULI_EXCEL, sheet_name="Overall")
fixation = "210"
non_final = "220"
item_codes = df_stimuli["W1_code"].unique().astype(str)
high_constraint = ("240", "241", "242", "243")  # just testing that both str and int work
low_constraint = ("244", "245", "246", "247")
emo_word = (240, 241, 244, 245)
neu_word = (242, 243, 246, 247)
conditions_list = [{},
                   {"high_constraint": high_constraint, "low_constraint": low_constraint},
                   {"emo_word": emo_word, "neu_word": neu_word}]

subj_epochs_list = []
for num in range(1, 28):
    raw_file_name = f"subj{str(num).zfill(3)}_after_ica_raw.fif"
    try:
        raw = mne.io.read_raw_fif(f"{DATA_DIR}/after_ica/{raw_file_name}", preload=True, verbose=False)
    except FileNotFoundError:
        continue

    print(f"subj{str(num).zfill(3)}")
    epochs_list = eeglab_logic_bin_epoch(raw.copy(),
                                         fixation=fixation,
                                         non_final=non_final,
                                         item_codes=item_codes,
                                         position_range=(1, 7),  # words 1 to 7
                                         conditions_list=conditions_list,
                                         tmin=-0.1, tmax=0.7,
                                         baseline=(-0.1, 0),
                                         reject_peak_to_peak={"eeg": 200e-6,
                                                              "eog": 200e-6},
                                         resample=None,
                                         preload=True,
                                         verbose=True)
    subj_epochs_list.append(epochs_list)
clear_output()


In [4]:
display(subj_epochs_list[0])

[<Epochs | 1451 events (all good), -0.1 – 0.7 s (baseline -0.1 – 0 s), ~310.4 MiB, data loaded,
  np.str_('w1'): 207
  np.str_('w2'): 208
  np.str_('w3'): 208
  np.str_('w4'): 208
  np.str_('w5'): 208
  np.str_('w6'): 208
  np.str_('w7'): 204>,
 <Epochs | 1451 events (all good), -0.1 – 0.7 s (baseline -0.1 – 0 s), ~310.4 MiB, data loaded,
  np.str_('w1/high_constraint'): 104
  np.str_('w1/low_constraint'): 103
  np.str_('w2/high_constraint'): 104
  np.str_('w2/low_constraint'): 104
  np.str_('w3/high_constraint'): 104
  np.str_('w3/low_constraint'): 104
  np.str_('w4/high_constraint'): 104
  np.str_('w4/low_constraint'): 104
  np.str_('w5/high_constraint'): 104
  np.str_('w5/low_constraint'): 104
  and 4 more events ...>,
 <Epochs | 1451 events (all good), -0.1 – 0.7 s (baseline -0.1 – 0 s), ~310.4 MiB, data loaded,
  np.str_('w1/emo_word'): 103
  np.str_('w1/neu_word'): 104
  np.str_('w2/emo_word'): 104
  np.str_('w2/neu_word'): 104
  np.str_('w3/emo_word'): 104
  np.str_('w3/neu_word

In [5]:
subj_drop_logs = [list(filter(lambda t: False if "IGNORED" in t or len(t) == 0 else True,
                              e[0].drop_log))  # changing index to 1 or 2 yields identical drop logs (of course, duh... LOL)
                              for e in subj_epochs_list]
how_many_dropped = [len(drop_log) for drop_log in subj_drop_logs]
print(how_many_dropped)

[1, 31, 24, 14, 15, 0, 2, 2, 0, 5, 0, 6, 0, 3, 0, 4, 1, 12, 3, 15]


In [6]:
subj_evokeds_list = [e[0].average(picks="eeg",
                                  method="mean",
                                  by_event_type=True) for e in subj_epochs_list]
subj_evokeds_list = [sorted(evokeds,
                            key=lambda e: int(re.findall(r"\d+", e.comment)[0])
                            ) for evokeds in subj_evokeds_list]
display(subj_evokeds_list[0])

[<Evoked | 'w1' (average, N=207), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>,
 <Evoked | 'w2' (average, N=208), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>,
 <Evoked | 'w3' (average, N=208), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>,
 <Evoked | 'w4' (average, N=208), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>,
 <Evoked | 'w5' (average, N=208), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>,
 <Evoked | 'w6' (average, N=208), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>,
 <Evoked | 'w7' (average, N=204), -0.1 – 0.7 s, baseline -0.1 – 0 s, 33 ch, ~251 KiB>]

In [7]:
avg_erp_data = np.stack([np.stack([e.get_data(units="uV") for e in evokeds], axis=0)
                                                          for evokeds in subj_evokeds_list], axis=0)
print(avg_erp_data.shape)
np.save(f"{DATA_DIR}/avg_erp/w1-w7_20subj.npy", avg_erp_data)

evoked = subj_evokeds_list[0][0]
f = open(f"{DATA_DIR}/avg_erp/channels.txt", "w")
f.writelines([ch + "\n" for ch in evoked.ch_names])
f.close()

(20, 7, 33, 801)


In [8]:
avg_erp_data = np.load(f"{DATA_DIR}/avg_erp/w1-w7_20subj.npy")
print(avg_erp_data.shape)

f = open(f"{DATA_DIR}/avg_erp/channels.txt", "r")
ch_names = f.readlines()
f.close()
ch_names = [ch.rstrip("\n") for ch in ch_names]
ch_name_to_idx_dict = {ch: idx for idx, ch in enumerate(ch_names)}

print(ch_name_to_idx_dict)

(20, 7, 33, 801)
{'FP1': 0, 'FP2': 1, 'F7': 2, 'F3': 3, 'FZ': 4, 'F4': 5, 'F8': 6, 'FT7': 7, 'FC3': 8, 'FCZ': 9, 'FC4': 10, 'FT8': 11, 'T7': 12, 'C3': 13, 'CZ': 14, 'C4': 15, 'T8': 16, 'M1': 17, 'TP7': 18, 'CP3': 19, 'CPZ': 20, 'CP4': 21, 'TP8': 22, 'M2': 23, 'P7': 24, 'P3': 25, 'PZ': 26, 'P4': 27, 'P8': 28, 'O1': 29, 'OZ': 30, 'O2': 31, 'FPZ': 32}


In [11]:
ch_roi = ["CZ", "C3", "C4", "CP3", "CP4", "CPZ"]
ch_idx = [ch_name_to_idx_dict[ch] for ch in ch_roi]

sfreq = 1000.0
epoch_tmin = -0.1
time_window = 0.25, 0.5
start = int(round((time_window[0] - epoch_tmin) * sfreq))
end = int(round((time_window[1] - epoch_tmin) * sfreq))

mean_amp = np.mean(avg_erp_data[:, :, ch_idx, start: end], axis=-1)
print(mean_amp.shape)

data_ols = np.concatenate(mean_amp, axis=0)
print(data_ols.shape)


(20, 7, 6)
(140, 6)


### Vanilla OLS on N400 time window (discarding subject differences, ROI channels fitted separately)

In [21]:
n_subj, n_words, n_channels = mean_amp.shape

pos = np.log(np.concatenate([np.arange(1, 8) for _ in range(n_subj)]))
ones = np.ones(pos.shape, dtype=float)
X = np.vstack([ones, pos]).T

print(X.shape)
print(X[:14])

(140, 2)
[[1.         0.        ]
 [1.         0.69314718]
 [1.         1.09861229]
 [1.         1.38629436]
 [1.         1.60943791]
 [1.         1.79175947]
 [1.         1.94591015]
 [1.         0.        ]
 [1.         0.69314718]
 [1.         1.09861229]
 [1.         1.38629436]
 [1.         1.60943791]
 [1.         1.79175947]
 [1.         1.94591015]]


In [22]:
channel_results = []
for i in range(n_channels):
    y = data_ols[:, i]
    ols = OLS(y, X)
    results = ols.fit(method="pinv") # uses Moore-Penrose pseudoinverse to solve the least squares
    channel_results.append(results)

In [26]:
coefs = [res.params for res in channel_results]
print(coefs)

[array([ 0.17309837, -0.50411946]), array([ 0.78768192, -0.91161311]), array([ 0.12238841, -0.70612539]), array([ 0.2882161 , -0.60149432]), array([-0.1842075 , -0.54108891]), array([-0.10091535, -0.34283874])]


### LME (adding subject as a random effect, ROI channels fitted separately)

In [ ]:
mean_amp.shape  # (n_subj, n_words, n_channels)

(20, 7, 6)

In [46]:
data_lme = mean_amp.transpose(2, 1, 0)
data_lme = np.concatenate(data_lme, axis=0).T  # for each subj, channels 1-6; for each channel, words 1-7
data_lme = np.concatenate(data_lme, axis=0)    
print(data_lme.shape)

subj_col = sum([[i]*(n_words*n_channels) for i in range(n_subj)], [])
channels_col = sum([[i]*n_words for i in range(n_channels)], [])
channels_col = sum([channels_col for _ in range(n_subj)], [])
pos_col = np.log(np.concatenate([np.arange(1, n_words+1)
                                      for _ in range(n_subj * n_channels)]))

df_data_lme = pd.DataFrame({"y": data_lme,
                            "subj": subj_col,
                            "channel": channels_col,
                            "position": pos_col})
df_data_lme["subj"] = df_data_lme["subj"].astype("category")
df_data_lme["channel"] = df_data_lme["channel"].astype("category")
display(df_data_lme)


(840,)


,y,subj,channel,position
0,-1.043419,0,0,0.000000
1,1.179987,0,0,0.693147
2,0.649458,0,0,1.098612
3,-2.368781,0,0,1.386294
4,-1.780581,0,0,1.609438
...,...,...,...,...
835,-0.800227,19,5,1.098612
836,-0.610725,19,5,1.386294
837,0.122499,19,5,1.609438
838,-1.315187,19,5,1.791759
